In [5]:
import pandas as pd
df = pd.read_csv("Life Expectancy Data.csv", encoding="latin1")

3. Preparação dos Dados

In [6]:
#removendo nomes com espaços
df.columns = df.columns.str.strip()
df.columns.tolist()

['Country',
 'Year',
 'Status',
 'Life expectancy',
 'Adult Mortality',
 'infant deaths',
 'Alcohol',
 'percentage expenditure',
 'Hepatitis B',
 'Measles',
 'BMI',
 'under-five deaths',
 'Polio',
 'Total expenditure',
 'Diphtheria',
 'HIV/AIDS',
 'GDP',
 'Population',
 'thinness  1-19 years',
 'thinness 5-9 years',
 'Income composition of resources',
 'Schooling']

In [7]:
#removendo países com apenas 10 registros
paises_validos = df['Country'].value_counts()
paises_validos = paises_validos[paises_validos == 16].index

df = df[df['Country'].isin(paises_validos)].copy()

df.shape

(2928, 22)

In [8]:
#qntd de linhas com paises desenvolvidos e em desenvolvimento
df['Status'].value_counts()

,count
Status,
Developing,2416
Developed,512


In [9]:
#% de linhas com paises desenvolvidos e em desenvolvimento
(df['Status'].value_counts(normalize=True) * 100).round(2)

,proportion
Status,
Developing,82.51
Developed,17.49


In [10]:
#valores ausentes dps da filtragem
df.isnull().sum().sort_values(ascending=False)

,0
Population,644
Hepatitis B,553
GDP,443
Total expenditure,226
Alcohol,193
Income composition of resources,160
Schooling,160
thinness 1-19 years,32
thinness 5-9 years,32
BMI,32


In [11]:
#% de valores ausentes dps da filtragem
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False).round(2)

,0
Population,21.99
Hepatitis B,18.89
GDP,15.13
Total expenditure,7.72
Alcohol,6.59
Income composition of resources,5.46
Schooling,5.46
thinness 1-19 years,1.09
thinness 5-9 years,1.09
BMI,1.09


In [12]:
#qntd de linhas com pelo menos um valor ausente
df.isnull().any(axis=1).sum()

np.int64(1279)

In [13]:
#qntd de linhas completas
df.dropna().shape

(1649, 22)

In [14]:
#valores ausentes por linha
df.isnull().sum(axis=1).value_counts().sort_index()

,count
0,1649
1,657
2,360
3,61
4,119
5,45
6,26
8,3
9,8


In [15]:
#tipo das colunas
df.dtypes

,0
Country,object
Year,int64
Status,object
Life expectancy,float64
Adult Mortality,float64
infant deaths,int64
Alcohol,float64
percentage expenditure,float64
Hepatitis B,float64
Measles,int64


In [16]:
#td sobre a coluna população
df['Population'].describe()

,Population
count,2.284000e+03
mean,1.276454e+07
std,6.103765e+07
min,3.400000e+01
25%,1.966738e+05
50%,1.391756e+06
75%,7.426746e+06
max,1.293859e+09


In [17]:
#colunas em listas
df.columns.tolist()

['Country',
 'Year',
 'Status',
 'Life expectancy',
 'Adult Mortality',
 'infant deaths',
 'Alcohol',
 'percentage expenditure',
 'Hepatitis B',
 'Measles',
 'BMI',
 'under-five deaths',
 'Polio',
 'Total expenditure',
 'Diphtheria',
 'HIV/AIDS',
 'GDP',
 'Population',
 'thinness  1-19 years',
 'thinness 5-9 years',
 'Income composition of resources',
 'Schooling']

In [18]:
#remove espaços duplicados nos nomes das colunas
df.columns = df.columns.str.replace(r'\s+', ' ', regex=True)

In [19]:
#colunas que serao analisadas
colunas_saude = [
    'Life expectancy',
    'Adult Mortality',
    'infant deaths',
    'Alcohol',
    'Hepatitis B',
    'Measles',
    'BMI',
    'under-five deaths',
    'Polio',
    'Total expenditure',
    'Diphtheria',
    'HIV/AIDS',
    'thinness 1-19 years',
    'thinness 5-9 years'
]

X = df[colunas_saude] #indicadores relacionados a ods 3
y = df['Status'] #resposta que queremos prever (Developed ou Developing)

print("Variáveis preditoras:", X.shape[1])
print("Variável-alvo:", y.name)


Variáveis preditoras: 14
Variável-alvo: Status


In [20]:
#verifica a qntd de valores ausentes em cada variável
X.isnull().sum().sort_values(ascending=False)

,0
Hepatitis B,553
Total expenditure,226
Alcohol,193
thinness 5-9 years,32
BMI,32
thinness 1-19 years,32
Diphtheria,19
Polio,19
Life expectancy,0
Adult Mortality,0


In [21]:
#preenche os valores ausentes com a mediana de cada coluna
X = X.fillna(X.median())

In [22]:
# Confere se ainda existem valores ausentes em X
X.isnull().sum().sum()

np.int64(0)

In [24]:
#confere as dimensões das variáveis preditoras e da variável-alvo
print("X:", X.shape)
print("y:", y.shape)

X: (2928, 14)
y: (2928,)


In [25]:
#exibe a proporção percentual de cada classe
(y.value_counts(normalize=True) * 100).round(2)

,proportion
Status,
Developing,82.51
Developed,17.49


(separando treino e teste)

In [26]:
from sklearn.model_selection import train_test_split

#divide os dados em 80% para treino e 20% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#confere as dimensões dos conjuntos
print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

Treino: (2342, 14)
Teste: (586, 14)


In [27]:
#confere a distribuição das classes nos dados de treino
print("Treino:")
print((y_treino.value_counts(normalize=True) * 100).round(2))

#confere a distribuição das classes nos dados de teste
print("\nTeste:")
print((y_teste.value_counts(normalize=True) * 100).round(2))

Treino:
Status
Developing    82.49
Developed     17.51
Name: proportion, dtype: float64

Teste:
Status
Developing    82.59
Developed     17.41
Name: proportion, dtype: float64


In [28]:
from sklearn.impute import SimpleImputer

#calcula as medianas usando somente os dados de treino
imputador = SimpleImputer(strategy='median')
X_treino = imputador.fit_transform(X_treino)

#aplica as medianas do treino aos dados de teste
X_teste = imputador.transform(X_teste)

In [29]:
import pandas as pd

#recupera os nomes das colunas nos conjuntos de treino e teste
X_treino = pd.DataFrame(X_treino, columns=X.columns)
X_teste = pd.DataFrame(X_teste, columns=X.columns)

#confere as dimensões e os nomes das colunas
print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)
print(X_treino.columns.tolist())


Treino: (2342, 14)
Teste: (586, 14)
['Life expectancy', 'Adult Mortality', 'infant deaths', 'Alcohol', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'thinness 1-19 years', 'thinness 5-9 years']


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd

#recupera os 14 indicadores originais, com os valores ausentes
colunas_saude = [
    'Life expectancy', 'Adult Mortality', 'infant deaths',
    'Alcohol', 'Hepatitis B', 'Measles', 'BMI',
    'under-five deaths', 'Polio', 'Total expenditure',
    'Diphtheria', 'HIV/AIDS', 'thinness 1-19 years',
    'thinness 5-9 years'
]

X = df[colunas_saude]
y = df['Status']

#divide os dados, preservando a proporção das classes
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#calcula as medianas somente com os dados de treino
imputador = SimpleImputer(strategy='median')
X_treino = pd.DataFrame(
    imputador.fit_transform(X_treino),
    columns=colunas_saude,
    index=X_treino.index
)

#aplica as mesmas medianas aos dados de teste
X_teste = pd.DataFrame(
    imputador.transform(X_teste),
    columns=colunas_saude,
    index=X_teste.index
)

#confere as dimensões e os valores ausentes
print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)
print("Ausentes no treino:", X_treino.isnull().sum().sum())
print("Ausentes no teste:", X_teste.isnull().sum().sum())


Treino: (2342, 14)
Teste: (586, 14)
Ausentes no treino: 0
Ausentes no teste: 0


4. Modelagem

In [31]:
#importa as ferramentas necessárias
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
import pandas as pd

#define os indicadores de saúde utilizados no projeto
colunas_saude = [
    'Life expectancy', 'Adult Mortality', 'infant deaths',
    'Alcohol', 'Hepatitis B', 'Measles', 'BMI',
    'under-five deaths', 'Polio', 'Total expenditure',
    'Diphtheria', 'HIV/AIDS', 'thinness 1-19 years',
    'thinness 5-9 years'
]

#separa os indicadores, o alvo e os países
X = df[colunas_saude]
y = df['Status']
grupos = df['Country']

#divide os dados por país, evitando que um país apareça nos dois conjuntos
divisor = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

indices_treino, indices_teste = next(
    divisor.split(X, y, groups=grupos)
)

X_treino = X.iloc[indices_treino]
X_teste = X.iloc[indices_teste]
y_treino = y.iloc[indices_treino]
y_teste = y.iloc[indices_teste]

#calcula as medianas apenas com os dados de treino
imputador = SimpleImputer(strategy='median')

X_treino = pd.DataFrame(
    imputador.fit_transform(X_treino),
    columns=colunas_saude,
    index=X_treino.index
)

#aplica as medianas do treino ao conjunto de teste
X_teste = pd.DataFrame(
    imputador.transform(X_teste),
    columns=colunas_saude,
    index=X_teste.index
)

#exibe as dimensões e a distribuição das classes
print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

print("\nClasses no treino:")
print(y_treino.value_counts())

print("\nClasses no teste:")
print(y_teste.value_counts())

print("\nPaíses presentes nos dois conjuntos:",
      len(set(grupos.iloc[indices_treino]) &
          set(grupos.iloc[indices_teste])))

Treino: (2336, 14)
Teste: (592, 14)

Classes no treino:
Status
Developing    1952
Developed      384
Name: count, dtype: int64

Classes no teste:
Status
Developing    464
Developed     128
Name: count, dtype: int64

Países presentes nos dois conjuntos: 0


(Decision Tree)

In [32]:
#importa o modelo e as métricas de avaliação
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

#cria e treina a árvore de decisão
modelo_arvore = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced'
)

modelo_arvore.fit(X_treino, y_treino)

#realiza previsões com os dados de teste
previsoes = modelo_arvore.predict(X_teste)

#avalia o desempenho do modelo
print("Acurácia:", round(accuracy_score(y_teste, previsoes), 4))

print("\nRelatório de classificação:")
print(classification_report(y_teste, previsoes))

print("\nMatriz de confusão:")
print(confusion_matrix(y_teste, previsoes))

Acurácia: 0.9223

Relatório de classificação:
              precision    recall  f1-score   support

   Developed       0.86      0.77      0.81       128
  Developing       0.94      0.97      0.95       464

    accuracy                           0.92       592
   macro avg       0.90      0.87      0.88       592
weighted avg       0.92      0.92      0.92       592


Matriz de confusão:
[[ 98  30]
 [ 16 448]]


(Random Forest)

In [33]:
#importa o modelo Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

#cria o modelo com pesos equilibrados entre as classes
modelo_floresta = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

#treina o modelo
modelo_floresta.fit(X_treino, y_treino)

#realiza previsões nos dados de teste
previsoes_floresta = modelo_floresta.predict(X_teste)

#exibe as métricas de avaliação
print("Acurácia:", round(
    accuracy_score(y_teste, previsoes_floresta), 4
))

print("\nRelatório de classificação:")
print(classification_report(y_teste, previsoes_floresta))

Acurácia: 0.8885

Relatório de classificação:
              precision    recall  f1-score   support

   Developed       0.83      0.61      0.70       128
  Developing       0.90      0.97      0.93       464

    accuracy                           0.89       592
   macro avg       0.86      0.79      0.82       592
weighted avg       0.88      0.89      0.88       592



5. Avaliação

In [34]:
#importa o Pandas para organizar os resultados
import pandas as pd

#obtém a importância de cada indicador no modelo treinado
importancias = pd.DataFrame({
    'Indicador': X_treino.columns,
    'Importância': modelo_arvore.feature_importances_
})

#ordena os indicadores do mais importante para o menos importante
importancias = importancias.sort_values(
    by='Importância',
    ascending=False
)

#exibe a importância dos indicadores
print(importancias.to_string(index=False))

          Indicador  Importância
    Life expectancy     0.528078
            Alcohol     0.220304
thinness 1-19 years     0.067752
  Total expenditure     0.034357
        Hepatitis B     0.033731
    Adult Mortality     0.024386
 thinness 5-9 years     0.021424
  under-five deaths     0.021404
                BMI     0.020614
            Measles     0.015351
         Diphtheria     0.004619
              Polio     0.004488
           HIV/AIDS     0.001824
      infant deaths     0.001668
